# Drive Your Dream Circuit: an AI-Storyboarded Onboard Lap (cloud)

We have a top-down map of a race track we designed. The task: generate an onboard video that drives one full lap of it.

| Chapter | Topic | What gets generated |
|---|---|---|
| 1 | The track map and the task | none |
| 2 | One prompt: 15 s, cannot finish the lap, hard to control | 1 video per run (optional) |
| 3 | Storyboard: split the lap into 7 shots | none |
| 4 | Keyframes: one onboard image at each shot boundary, checked against the storyboard, then the car made consistent | 8 images, plus 1 per revision; the car edit runs a local model and needs a GPU (bundled results available) |
| 5 | Shots: first-last-frame video, one shot at a time | up to 14 videos (bundled results available) |
| 6 | Stitch the film and find the problems | none |
| 7 | Fix the T7–T8 chicane with the storyboard | 2 keyframes, 3 videos × 2 takes (bundled results available) |
| 8 | Wrap-up and exercise | none |

This notebook only sets up the environment, starts ComfyUI, prepares the media and stitches the films; images and videos are generated in ComfyUI in the browser.
Every generation step ships with ready-made keyframes and videos, so you can stitch the film quickly from them or generate everything yourself.
The chapter outlines in `docs/` and the notes inside the ComfyUI workflows are in Chinese; this notebook covers the same points.

### How to use this notebook

- Run the cells from top to bottom: click a cell and press `Shift+Enter`. `[*]` on the left means it is still running; a number means it is done, and its output appears below the cell.
- Two routes, chosen per chapter:
  - **Route A, bundled media**: run every cell in order; the bundled media fill in the keyframes and videos.
  - **Route B, generate yourself**: at each "In the browser" step, generate in ComfyUI first, then run the picker cell (for videos, fill in the takes you chose first); anything you did not generate is still filled in from the bundled media.
- After a kernel restart, or when you come back the next day, rerun section 0; finished steps are skipped.

## 0. Setup

In [ ]:
import os, sys, shutil, subprocess, urllib.request
from getpass import getpass
from pathlib import Path

PROJECT = Path.cwd().parent
if not (PROJECT / "scripts" / "stitch_lap.py").is_file():
    PROJECT = Path.home() / "dream-circuit-lap"
    if not PROJECT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/wangxunx/dream-circuit-lap.git", str(PROJECT)], check=True)
COMFY = Path(os.environ.get("COMFY_DIR", "/comfyui_workspace/ComfyUI"))
PY = sys.executable
assert (COMFY / "main.py").is_file(), f"ComfyUI not found at {COMFY}, set COMFY_DIR"
os.environ["COMFYUI_DIR"] = str(COMFY)
print("project:", PROJECT)
print("ComfyUI:", COMFY)

In [ ]:
%pip install -q -r {PROJECT}/requirements.txt

### Install the two cloud nodes

[ComfyUI-AMD-Qwen-Image-2.1](https://github.com/wangxunx/ComfyUI-AMD-Qwen-Image-2.1) generates the keyframes and [ComfyUI-AMD-MiniMaxH3](https://github.com/wangxunx/ComfyUI-AMD-MiniMaxH3) the videos. Nodes that are already installed are skipped.

In [ ]:
for repo in ("ComfyUI-AMD-Qwen-Image-2.1", "ComfyUI-AMD-MiniMaxH3"):
    if (COMFY / "custom_nodes" / repo).is_dir():
        print("installed:", repo)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/wangxunx/{repo}.git", str(COMFY / "custom_nodes" / repo)], check=True)
        print("cloned:", repo)

### Configure the H3 key

Register at https://h3.oneclickamd.ai to get a key. It is written to ComfyUI's `user/h3_gateway_api_key.txt` and never ends up in workflows or video metadata.
Running the next cell shows an input box below it: paste the key and press Enter.

In [ ]:
key_file = COMFY / "user" / "h3_gateway_api_key.txt"
if os.environ.get("H3_GATEWAY_API_KEY") or key_file.is_file():
    print("H3 key already configured")
else:
    key_file.parent.mkdir(parents=True, exist_ok=True)
    key_file.write_text(getpass("H3 key: ").strip())
    print("saved to", key_file)

### Start ComfyUI

The script starts ComfyUI in the background, prints the `ComfyUI URL` and returns, so you can keep running the cells below. Rerunning this cell restarts ComfyUI and keeps the URL.

> **Note:** anyone with this URL can queue jobs on your H3 key. Do not post it in group chats or anywhere public.

In [ ]:
!COMFY_DIR="{COMFY}" bash "{PROJECT}/scripts/start_comfyui_cloud.sh"

### Prepare media

Download the historical results (about 72 MB, sha256-checked, safe to rerun), plus Noto Sans SC if the system has no CJK font; the stitching script draws its labels with it.
Then install the chapter workflows into ComfyUI: `ch2_single_prompt`, `dream7_*` and `dream8_*` appear in the **Workflows** panel on the left of the browser.

In [ ]:
!"{PY}" "{PROJECT}/scripts/fetch_media.py" --fonts
(COMFY / "user" / "default" / "workflows").mkdir(parents=True, exist_ok=True)
shutil.copy(PROJECT / "workflows" / "ch2_single_prompt.json", COMFY / "user" / "default" / "workflows")
shutil.copy(PROJECT / "assets" / "fictional_track_v1.png", COMFY / "input")
!"{PY}" "{PROJECT}/scripts/make_workflows.py" --install
!"{PY}" "{PROJECT}/scripts/make_workflows.py" --shots lap_8shots.json --install

### First time in ComfyUI

1. Open ComfyUI in the browser (http://127.0.0.1:8188 for the local notebook, the `ComfyUI URL` printed above for the cloud one). A Templates dialog appears on first launch; close it with the × in the top right. The middle is the canvas: boxes are nodes, and links show where the data flows.
2. Press `w` or click **Workflows** on the left, then click a name to open that workflow. If this project's workflows are not listed, reload the page (F5).
3. Every workflow in this project has a yellow note on its left saying what the step does and what to prepare first (in Chinese). Scroll to zoom, drag an empty spot to pan, press `.` to fit all nodes.
4. Click the blue **Run** button in the top right to queue the job; "N active" next to it counts the jobs still running. Generation happens in the cloud; a video usually takes a few minutes.
5. Results show up in the Save Image / Save Video nodes and are saved in ComfyUI's `output/` directory.
6. A Load Image node outlined in red with a "Media input missing" or "errors found" toast in the top right: the image it reads is not in `input/` yet. Run the matching copy or picker cell in this notebook first, then reload the page (F5) and the red outline goes away.
7. Do not change the filename prefixes in the chapter 4–7 workflows; the picker and stitching cells find files by these names.

`play` below references videos by relative path, so open this notebook in JupyterLab / JupyterHub with the project directory under Jupyter's root directory; images are shrunk and embedded directly.

In [ ]:
from IPython.display import HTML, Markdown, display
from base64 import b64encode
from io import BytesIO
from PIL import Image
import json

MEDIA = PROJECT / "media"

def figure(inner, p, label, width):
    caption = f'{label}<br><small><code>{Path(p).name}</code></small>' if label else Path(p).name
    return f'<figure style="display:inline-block;vertical-align:top;margin:4px">{inner}<figcaption style="max-width:{width}px">{caption}</figcaption></figure>'

def play(*paths, width=480, labels=()):
    labels = labels or [None] * len(paths)
    display(HTML("".join(figure(f'<video src="{os.path.relpath(p)}" width="{width}" controls muted></video>', p, label, width) for p, label in zip(paths, labels))))

def show(*paths, width=360, labels=()):
    tags = []
    for p, label in zip(paths, labels or [None] * len(paths)):
        im = Image.open(p).convert("RGB")
        im.thumbnail((width * 2, width * 2))
        buf = BytesIO()
        im.save(buf, "JPEG", quality=85)
        tags.append(figure(f'<img src="data:image/jpeg;base64,{b64encode(buf.getvalue()).decode()}" width="{width}">', p, label, width))
    display(HTML("".join(tags)))

def copy_missing(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy(src, dst)
        print("copied", dst)

def pick(patterns, src_dir, dst):
    missing = []
    for name, pattern in patterns.items():
        found = sorted(Path(src_dir).glob(pattern), key=os.path.getmtime)
        if not found:
            missing.append(name)
            continue
        target = Path(str(dst).format(name))
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(found[-1], target)
        print(found[-1].name, "->", target)
    if missing:
        print("not generated yet:", " ".join(missing))

## Chapter 1: the track map and the task

5.1 km, 11 turns, anticlockwise. From the start/finish straight: the heavy-braking T1–T2, the climbing turns T3–T5, the fast sweeper T6, a short straight into the T7–T8 chicane, the downhill T9 and the final sector T10–T11.

The task: from this map, generate an onboard video that drives one full lap. Outline (in Chinese): `docs/ch1_map.md`.

In [ ]:
show(PROJECT / "assets" / "fictional_track_v1.png", width=900)

## Chapter 2: one prompt for the whole lap

The most direct idea: use the top-down map as the first frame, describe the 7 track elements in one prompt, and let MiniMax H3 generate.
Open the `ch2_single_prompt` workflow in the browser to try it (1 video per run). Below are the 9 attempts from the time:

- Attempts 1–6 kept refining the prompt: stress anticlockwise, say where the start is, ask for Chinese narration. Every video still flies into the map, the "onboard view" stays framed between the map's legend and elevation profile, 15 seconds only covers a short stretch, and the turns do not match the map.
- Attempts 5 and 6 used exactly the same prompt and came out with almost the same structure, differing only in details: rerunning does not help, because the input decides the result.
- Attempts 7–8 narrowed the goal to "only the start/finish straight up to T1", but the style turned cartoonish; attempt 9 added "road surface and surroundings consistent with the map" and finally got a full-screen, photorealistic cockpit view.

Two lessons: shoot one short stretch at a time, and the first frame should be an onboard image, not the map. That is where storyboards and keyframes come from.

In [ ]:
play(MEDIA / "ch2_single_prompt_01.mp4", MEDIA / "ch2_single_prompt_05.mp4", MEDIA / "ch2_single_prompt_06.mp4", width=360)
play(MEDIA / "ch2_single_prompt_07.mp4", MEDIA / "ch2_single_prompt_09.mp4", width=360)

## Chapter 3: the storyboard

Split the lap into 7 shots of 5–8 seconds each. Neighbouring shots share a keyframe at their boundary, K0–K7: shot i starts at K(i-1) and ends at K(i), so the shots join up.
The storyboard is `storyboards/lap_7shots.json`; the generated guide is `docs/storyboard_dream7.md` (in Chinese).

In [ ]:
sb = json.load(open(PROJECT / "storyboards" / "lap_7shots.json", encoding="utf-8"))
rows = ["| Shot | Sector | First → last frame | Length | Turns |", "|---|---|---|---|---|"]
rows += [f"| {s['id']} | {s['title_en']} | {s['first']} → {s['last']} | {s['duration']} s | {s['turns_en']} |" for s in sb["shots"]]
display(Markdown("\n".join(rows)))
show(PROJECT / "assets" / "storyboard_map_7shots_en.png", width=1000)

Each colour is the stretch one shot drives, and the circles mark the keyframes. Come back to this map whenever later chapters mention K0–K7, shots 1–7, or `dream7_shot_01` to `dream7_shot_07`.

## Chapter 4: keyframes

Keyframes are the "contract" between shots: the video starts at the first frame and must arrive at the last one, so the model drives toward whatever the last frame shows. Check every image against the storyboard; the most important check is that the road ahead bends the same way as on the map.

The nodes have no random seed: with an unchanged prompt, generating again gives the same image. So generate each keyframe once; when it is wrong, find where it disagrees with the storyboard, write that into the scene description and generate again. Below are several generations of the same K0, each fixing one problem:

1. The far end of the straight bends left, then right, then climbs; from the start/finish line the map shows only one left-hander, T1. The scene was changed to a dead-straight straight ending in a single left-hander, flat all around.
2. The left-hander is too close, so the straight looks only two or three hundred metres long; on the map it is about 1 km. The scene was changed so the straight runs to the vanishing point, with the left-hander only faintly visible there.
3. The road is right, but the start light gantry is far behind the red car and the red lights are just a few dots. The scene was changed to put the gantry about 20 m ahead of the car, filling the top of the frame.
4. Final.

The gantry in the finishing K7 must be the same one as in K0, so K7's scene description spells out K0's gantry piece by piece.

In [ ]:
show(*[PROJECT / "examples" / "k0_versions" / f"K0_v{v}.jpg" for v in range(1, 5)], width=440)

In the browser:

1. Open `dream7_keyframes_pilot` and click Run to generate only K0 and K1 first; check the style and camera position.
2. Open `dream7_keyframes` and click Run to generate all 8, one row each.
3. Check each one against the chapter 3 storyboard table and map: which way the road bends, how far away the turn is, whether elements such as the light gantry and grandstands are there, and whether the camera and car match.
4. If one is wrong, edit the scene description at the start of that node's prompt (the part before `Photorealistic onboard`) and click Run again. ComfyUI only regenerates the edited image; after a ComfyUI restart everything regenerates, so first select the Save Image nodes you do not want redone and press Ctrl+M to mute them.
5. When all are good, run the next cell: the newest version of each keyframe is copied to ComfyUI's `input/dream7_K0.png` to `dream7_K7.png`.

For route A just run the next two cells: the picker does nothing if it finds none of your images, and the cell after it fills in the missing keyframes from the bundled ones without overwriting files you picked. Outline (in Chinese): `docs/ch4_keyframes.md`.

In [ ]:
pick({f"K{i}": f"K{i}_*.png" for i in range(8)}, COMFY / "output" / "dream7", COMFY / "input" / "dream7_{}.png")

In [ ]:
for i in range(8):
    copy_missing(PROJECT / "examples" / "keyframes" / f"dream7_K{i}.png", COMFY / "input" / f"dream7_K{i}.png")
show(*[COMFY / "input" / f"dream7_K{i}.png" for i in range(8)], width=300)

### Make the car consistent

`style` describes the car, but text-to-image still draws a different car in each keyframe: the part on top of the halo changes size, some bodies have red or white on them, the camera sits higher or lower.
When the first and last frame show two different cars, the video swaps cars midway, and neighbouring shots share keyframes, so this step comes before the shots.

The edit runs the local Qwen Image 2.1 model (about 17 GB, needs a GPU).

In the browser (route B):

1. Set `DOWNLOAD` in the next cell to `True` and run it to download the missing models (existing ones are skipped), then restart ComfyUI or reload the page.
2. Open `dream7_keyframes_edit` and click Run: with K1 as the reference, the car in K0 and K2–K7 becomes K1's car.
3. Check each one: the same car as K1 at the same camera height, with the road, the red car and the grandstands unchanged. If one is off, add a sentence to its instruction or change the KSampler seed and run again.
4. When you are happy, run the cell after next: the edits overwrite the keyframes in `input/`. Then reload the page.

For route A just run the cells below: no models are downloaded and the picker does nothing. The bundled keyframes already have the consistent car and the bundled videos were shot from them; the last cell shows the edit before and after. Outline (in Chinese): `docs/ch4_keyframes.md`.

In [ ]:
DOWNLOAD = False  # set to True for route B
for folder, name in (("diffusion_models", "qwen_image_2.1_int8_convrot.safetensors"),
                     ("text_encoders", "qwen3vl_8b_int8_convrot.safetensors"),
                     ("vae", "qwen_image_2.1_vae_bf16.safetensors")):
    dst = COMFY / "models" / folder / name
    if dst.exists():
        print("found", dst)
    elif DOWNLOAD:
        part = dst.with_name(name + ".part")
        urllib.request.urlretrieve(f"https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/{folder}/{name}", part)
        part.rename(dst)
        print("downloaded", dst)
    else:
        print("missing", dst)

In [ ]:
pick({f"K{i}": f"K{i}_edit_*.png" for i in (0, 2, 3, 4, 5, 6, 7)}, COMFY / "output" / "dream7", COMFY / "input" / "dream7_{}.png")

In [ ]:
show(PROJECT / "examples" / "car_edit" / "K1_reference.jpg", width=440)
show(*[PROJECT / "examples" / "car_edit" / f"{k}_{v}.jpg" for k in ("K0", "K2", "K6") for v in ("before", "after")], width=440)

## Chapter 5: shooting the shots

Shot i goes to the MiniMax H3 first-last-frame node with K(i-1) as the first frame and K(i) as the last frame; the prompt only describes the motion.

In the browser:

0. Finish chapter 4 first (the keyframes are in `input/`), then reload the page once, otherwise the Load Image nodes turn red.
1. Open `dream7_shot_01` for shot 1 (the pilot). Each Run makes 2 takes: the top one is `_t1`, the bottom one `_t2`. Check the style, whether the start and end are close to K0 and K1, and whether the direction and sense of speed are right.
2. When you are happy, open `dream7_shot_02` to `dream7_shot_07` in turn and click Run. You can submit them back to back; the jobs queue.
3. Fill in the take you chose for each shot in `SHOT_TAKES` in the next cell and run it; the chosen videos are copied to `output/video/dream7/selected/shot_0X.mp4`.

What to look for when picking: does it turn the right way, are there cuts or reversing, do the car or camera change midway, are the start and end close to the keyframes.
For route A just run the next two cells; the bundled videos fill in missing shots without overwriting your picks. The players below show shots 1 and 2 from the bundled media. Outline (in Chinese): `docs/ch5_shots.md`.

In [ ]:
SHOT_TAKES = {"shot_01": 1, "shot_02": 1, "shot_03": 1, "shot_04": 1, "shot_05": 1, "shot_06": 1, "shot_07": 1}
pick({s: f"{s}_t{t}_*" for s, t in SHOT_TAKES.items()}, COMFY / "output" / "video" / "dream7", COMFY / "output" / "video" / "dream7" / "selected" / "{}.mp4")

In [ ]:
for i in range(1, 8):
    copy_missing(MEDIA / f"dream7_shot_0{i}.mp4", COMFY / "output" / "video" / "dream7" / "selected" / f"shot_0{i}.mp4")
play(MEDIA / "dream7_shot_01.mp4", MEDIA / "dream7_shot_02.mp4", width=480)

## Chapter 6: stitch the film and find the problems

`stitch_lap.py` joins the 7 shots in order with crossfades at the seams, overlays sector titles and a minimap in the lower right (the red dot moves at each shot's speed), and adds an intro and an outro. `--en` gives English titles.

While watching, note: where does the steering disagree with the map? Where does the picture jump? Does the minimap's red dot match the picture?
Sort the problems into four layers, storyboard, keyframes, prompts and stitching, which correspond to chapters 3 to 6. Outline (in Chinese): `docs/ch6_problems.md`.

In [ ]:
!"{PY}" "{PROJECT}/scripts/stitch_lap.py" --en
play(PROJECT / "output" / "dream_circuit_7shots_en.mp4", width=900)

## Chapter 7: fix the T7–T8 chicane with the storyboard

On the map T7 is a right-hander and T8 a left-hander, but shot 5 of the chapter 6 film shows only one left-hander: T7 is gone.
Now let's fix it step by step: each step says what is changed and in which layer, then shows the result. The videos and images are already generated, so you only need to watch.

### Original shot 5: K4 → K5 drives T7–T8 in one shot

K4 is the view leaving T6 and K5 the view leaving the chicane; their headings differ by only about 15°. Driving from K4 to K5, the easiest path for the model is a slight drift, so the T7 right turn is simply skipped.

Watch: only one left-hander.

In [ ]:
play(MEDIA / "dream7_shot_05.mp4", width=480, labels=["Original shot 5: only one left-hander"])

### Step 1: change the prompt only (prompt layer, chapter 5)

Shot and keyframes stay the same; the prompt now stresses "right, then left".

Watch: the T7 right-hander is still barely visible. When the problem is in the storyboard, changing the prompt does not help.

In [ ]:
play(MEDIA / "chicane_1_prompt_only.mp4", width=480, labels=["Prompt only: T7 is still barely visible"])

### Step 2: split into two shots (storyboard layer, chapter 3)

One shot for T7 and one for T8, with a new boundary keyframe K5 at the T7 exit, so each shot turns only once.

Watch: the T7 shot starts by turning left. The cause is the old K4 on the left: the end of the short straight shows a left-hander, so the model drives left.

In [ ]:
show(PROJECT / "examples" / "keyframes" / "dream7_K4.png", width=360, labels=["Old K4: a left-hander at the end of the straight"])
play(MEDIA / "chicane_2_split.mp4", width=480, labels=["T7 shot after the split: starts by turning left"])

### Step 3: redraw K4 (keyframe layer, chapter 4)

K4 is redrawn so the end of the short straight shows only the T7 right-hander.

Watch: the start is right now, turning right first. But the first K5 shows both the right- and the left-hander, and the picture jumps within a second, as if replaying T7–T8.

In [ ]:
show(PROJECT / "examples" / "keyframes" / "dream8_K4.png", MEDIA / "chicane_K5_first_try.png", width=360, labels=["New K4: only the T7 right-hander", "First K5: both the right- and the left-hander"])
play(MEDIA / "chicane_3_new_K4.mp4", width=480, labels=["New K4 → first K5: jumps within a second"])

### Step 4: redraw K5 (keyframe layer, chapter 4)

K5 is redrawn just after T7, with only the T8 left-hander ahead. The T8 shot follows the same rule: K5 shows only the left-hander, K6 only the downhill.

Watch: shot 5 is a full T7 right-hander and shot 6 follows with the T8 left-hander. Fixed.

In [ ]:
show(PROJECT / "examples" / "keyframes" / "dream8_K5.png", width=360, labels=["New K5: only the T8 left-hander"])
play(MEDIA / "dream8_shot_05.mp4", MEDIA / "dream8_shot_06.mp4", width=480, labels=["New shot 5 (T7): a full right-hander", "New shot 6 (T8): the left-hander"])

### Two rules

1. **One change of direction per shot.** Put two turns in one shot and the model cuts the corner, skipping one.
2. **A boundary keyframe shows only the next turn.** The model drives toward whatever the far end of the keyframe shows; show two turns and the shot repeats or jumps.

### Stitch the 8-shot version

The 8-shot storyboard `storyboards/lap_8shots.json` redoes only keyframes K4 and K5 and shots 4–6; the other keyframes come from chapter 4 and the other shots from chapter 5.
The old shot 5 is split into shot 5 (T7) and shot 6 (T8) with a new K5 between them, so everything after it shifts by one: the 7-shot K5, K6 and K7 are K6, K7 and K8 here, and the old shots 6 and 7 are shots 7 and 8. Compare with the map below.
Route B: in the browser, open `dream8_keyframes` to generate K4 and K5 and check them against the storyboard (if one is wrong, edit its scene description and generate again), run the picker cell below and reload the page. Next open `dream8_keyframes_edit` to make the car in K4 and K5 match `dream7_K1.png`, check them, run the picker cell again (it takes the newest version of each keyframe, which is the edit) and reload the page. Then open `dream8_shot_04` to `dream8_shot_06` to generate the videos, come back, change the video take number in the picker cell (the 1 in `_t1_`) to the ones you chose, and run it again.
For route A just run the cells below. At the end you stitch the 8-shot film and compare it with the chapter 6 version.

In [ ]:
show(PROJECT / "assets" / "storyboard_map_8shots_en.png", width=1000)

In [ ]:
pick({k: f"{k}_*.png" for k in ("K4", "K5")}, COMFY / "output" / "dream8", COMFY / "input" / "dream8_{}.png")
pick({s: f"{s}_t1_*" for s in ("shot_04", "shot_05", "shot_06")}, COMFY / "output" / "video" / "dream8", COMFY / "output" / "video" / "dream8" / "selected" / "{}.mp4")

In [ ]:
for k in ("K4", "K5"):
    copy_missing(PROJECT / "examples" / "keyframes" / f"dream8_{k}.png", COMFY / "input" / f"dream8_{k}.png")
for i in (4, 5, 6):
    copy_missing(MEDIA / f"dream8_shot_0{i}.mp4", COMFY / "output" / "video" / "dream8" / "selected" / f"shot_0{i}.mp4")

In [ ]:
!"{PY}" "{PROJECT}/scripts/stitch_lap.py" --shots lap_8shots.json --en
play(PROJECT / "output" / "dream_circuit_8shots_en.mp4", width=900)

## Chapter 8: wrap-up and exercise

- One prompt suits one action, not a whole lap; split the task into shots, each doing one thing.
- Keyframes are the contract between shots: the model drives toward whatever they show.
- When something goes wrong, classify it first (storyboard, keyframes, prompts, stitching), then fix it in that layer.

Exercise: the film still has a problem where the downhill straight meets T9 (it cuts into T9 before the straight ends). Use the same method to split shot 7, redraw the boundary keyframe, and stitch your own version. Outline (in Chinese): `docs/ch8_wrapup.md`.

## Finish

When you are done, stop ComfyUI and the tunnel; the public URL stops working.

In [ ]:
!bash "{PROJECT}/scripts/start_comfyui_cloud.sh" stop